# Parlay pricing and the correlation assumption

[Open in Colab](https://colab.research.google.com/github/JacobiusMakes/parlayapi-notebooks/blob/main/05-parlay-pricing.ipynb)

Run the existing local independent-price calculations and read their correlation caveats.
Live data and the optional API cross-check require explicit choices.

**Run All is offline by default:** no API calls or key prompts. Existing numerical examples
are illustrative calculations, not current sportsbook observations. Select `demo` explicitly
for a limited anonymous sample. Select `account` only in a private copy for requests using
your own key and allowance. The hidden key prompt appears when an account request runs.
See [current docs](https://parlay-api.com/docs) and [plans](https://parlay-api.com/pricing).


In [ ]:
# Run All is offline by default. Choose demo or account explicitly for API calls.
MODE = "offline"  # "offline", "demo", or "account"
SPORT = "baseball_mlb"
RUN_EXTRA_API_CHECKS = False
RUN_POLLING = False
SAVE_PRIVATE_CSV = False

import getpass
import json
import requests

SPORTS = {"baseball_mlb", "basketball_nba", "americanfootball_nfl",
          "icehockey_nhl", "soccer_epl", "mma_mixed_martial_arts"}
BASE_URL = "https://parlay-api.com"
_runtime_key = None

def get_runtime_key():
    global _runtime_key
    if MODE != "account":
        raise RuntimeError("Choose account mode before entering a key.")
    if _runtime_key is None:
        value = getpass.getpass("Your own ParlayAPI key (hidden; account credits apply): ")
        if not value or any(ord(c) < 33 or ord(c) > 126 for c in value):
            raise RuntimeError("Enter a valid key without whitespace or control characters.")
        _runtime_key = value
    return _runtime_key

def request_json(path, *, params=None, method="GET", body=None, account=False):
    if MODE not in {"offline", "demo", "account"} or SPORT not in SPORTS:
        raise RuntimeError("Choose a listed mode and supported sport.")
    if MODE == "offline":
        raise RuntimeError("Offline mode makes no API requests.")
    if not path.startswith("/") or path.startswith("//") or ".." in path or "\\" in path:
        raise RuntimeError("Use a fixed API path.")
    headers = {"Accept": "application/json"}
    if account:
        headers["X-API-Key"] = get_runtime_key()
    try:
        with requests.request(method, "https://parlay-api.com" + path,
                              params=params, json=body, headers=headers,
                              timeout=30, allow_redirects=False, stream=True) as response:
            if response.status_code != 200:
                raise RuntimeError(f"API returned HTTP {response.status_code}. No automatic retry.")
            chunks = []
            size = 0
            for chunk in response.iter_content(65536):
                size += len(chunk)
                if size > 10_000_000:
                    raise RuntimeError("Response exceeds the size limit.")
                chunks.append(chunk)
            return json.loads(b"".join(chunks))
    except (requests.RequestException, ValueError):
        raise RuntimeError("Request or JSON response failed. No automatic retry.") from None

if MODE not in {"offline", "demo", "account"} or SPORT not in SPORTS:
    raise RuntimeError("Choose a listed mode and supported sport.")
print("Mode:", MODE)
print("Offline runs the math without network. Demo is a limited anonymous sample.")
print("Account mode prompts at runtime and uses your own allowance. Keep that copy private.")


In [ ]:
# American <-> decimal conversions, matching the conventions used by
# https://parlay-api.com/tools/no-vig-calculator and /tools/parlay-calculator.

def american_to_decimal(a):
    """+150 -> 2.5, -110 -> 1.9091. Valid American odds are >= +100 or <= -100."""
    a = float(a)
    if abs(a) < 100:
        raise ValueError(f"{a} is not a valid American price (must be >= +100 or <= -100)")
    if a > 0:
        return 1 + a / 100
    return 1 + 100 / (-a)

def decimal_to_american(d):
    """2.5 -> +150, 1.9091 -> -110 (rounded to the nearest integer)."""
    d = float(d)
    if d <= 1:
        raise ValueError(f"decimal odds must be > 1, got {d}")
    if d >= 2:
        return round((d - 1) * 100)
    return round(-100 / (d - 1))

def implied_prob(decimal_odds):
    """Implied win probability of decimal odds (includes the vig)."""
    return 1.0 / float(decimal_odds)

def fmt_american(a):
    return ("+" if a > 0 else "") + str(int(a))

## The core math

Convert every leg to decimal odds, multiply the decimals, multiply by the stake:

```
combined_dec = dec_1 * dec_2 * ... * dec_n
payout       = stake * combined_dec      (includes your stake back)
profit       = payout - stake
implied_prob = 1 / combined_dec
```

In [ ]:
from functools import reduce

def parlay(american_legs, stake=100.0):
    decs = [american_to_decimal(a) for a in american_legs]
    combined = reduce(lambda x, y: x * y, decs, 1.0)
    return {
        "legs": american_legs,
        "combined_decimal": combined,
        "combined_american": decimal_to_american(combined),
        "implied_prob": 1.0 / combined,
        "stake": stake,
        "payout": stake * combined,
        "profit": stake * combined - stake,
    }

p = parlay([-110, -110, -110], stake=100)
print(f"three legs at -110, $100 stake:")
print(f"  combined decimal  {p['combined_decimal']:.4f}")
print(f"  combined American {fmt_american(p['combined_american'])}")
print(f"  implied prob      {p['implied_prob']*100:.2f}%")
print(f"  payout ${p['payout']:.2f}  (profit ${p['profit']:.2f})")

# The canonical case from the site parlay calculator:
assert abs(p["combined_decimal"] - 6.9579) < 1e-3
assert p["combined_american"] == 596
assert abs(p["payout"] - 695.79) < 0.01
assert abs(p["implied_prob"] - 0.1437) < 1e-3
print("matches the /tools/parlay-calculator canonical case (+596, $695.79, 14.37%)")

## The vig compounds with every leg

Each -110 price carries roughly a 4.5% edge against the bettor: its fair value is
+100 in a balanced market (notebook 02), so if a -110 leg's true chance is 50%, a
$1 bet returns `0.50 * 1.9091 = 0.9545` in expectation. Because parlay legs
multiply, that edge compounds: n such legs return `0.9545 ** n`.


In [ ]:
leg_dec = american_to_decimal(-110)
true_p = 0.50  # assume the balanced-market fair value
per_leg_return = true_p * leg_dec

print("legs | expected return per $1 | expected loss")
for n in range(1, 7):
    er = per_leg_return ** n
    print(f"  {n}  |        {er:.4f}          |   {(1-er)*100:5.1f}%")

three = per_leg_return ** 3
assert abs(three - 0.870) < 5e-4
print("\nthree legs of -110 at a true 50% each: expected return 0.870,")
print("about a 13% expected loss, exactly as the site calculator's FAQ works out.")

## Build a parlay from the live board

Two legs, one from each of the first two events on the slate, taking each side's
best available price across books (line shopping matters even more in parlays,
because the improvement multiplies).

In [ ]:
def fetch_odds(sport=None, markets="h2h,spreads,totals", odds_format="american"):
    """A chosen demo or account request. Offline returns no live observations."""
    sport = sport or SPORT
    if sport not in SPORTS or odds_format != "american":
        raise RuntimeError("Choose a supported sport and American odds.")
    requested = markets.split(",")
    if not requested or any(m not in {"h2h", "spreads", "totals"} for m in requested):
        raise RuntimeError("Choose h2h, spreads, or totals.")
    if MODE == "offline":
        return []
    if MODE == "account":
        events = request_json(f"/v1/sports/{sport}/odds", account=True,
                              params={"markets": markets,
                                      "oddsFormat": "american"})
    else:
        payload = request_json(f"/v1/try/{sport}/odds")
        if (not isinstance(payload, dict) or payload.get("demo") is not True
                or not isinstance(payload.get("events"), list) or len(payload["events"]) > 5):
            raise RuntimeError("Unexpected demo response. No observations used.")
        events = payload["events"]
    if not isinstance(events, list) or any(not isinstance(e, dict) or e.get("sport_key") != sport for e in events):
        raise RuntimeError("Response does not match the chosen sport.")
    return events


In [ ]:
try:
    events = fetch_odds(markets="h2h")
except Exception:
    events = []
    print(f"Fetch failed (request failed). Check your connection or key and re-run this cell.")
legs = []
for ev in events[:2]:
    quotes = []
    for bm in ev.get("bookmakers", []):
        for mkt in bm.get("markets", []):
            if mkt["key"] != "h2h":
                continue
            for out in mkt["outcomes"]:
                if out["name"] == ev["home_team"]:
                    quotes.append((bm["key"], out["price"],
                                   american_to_decimal(out["price"])))
    if not quotes:
        continue
    # Outlier guard: a thin exchange-style listing can post a price no real
    # book will honor, and a naive max would pick exactly that row. Drop
    # anything more than 15% above the median decimal before shopping.
    med = sorted(q[2] for q in quotes)[len(quotes) // 2]
    sane = [q for q in quotes if q[2] <= med * 1.15] or quotes
    book, price, _ = max(sane, key=lambda q: q[2])
    legs.append({"event": f"{ev['away_team']} at {ev['home_team']}",
                 "side": ev["home_team"], "book": book, "price": price})

if len(legs) < 2:
    print("Fewer than two priced events loaded. Offline does not fetch data; choose demo")
    print("explicitly to request a sample, or change SPORT in the config cell.")
else:
    for leg in legs:
        print(f"leg: {leg['side']} {fmt_american(leg['price'])} "
              f"(best price, at {leg['book']}) in {leg['event']}")
    p = parlay([leg["price"] for leg in legs], stake=50)
    print(f"\ncombined: {p['combined_decimal']:.4f} decimal "
          f"= {fmt_american(p['combined_american'])} American")
    print(f"$50 pays ${p['payout']:.2f} if both legs win "
          f"(implied {p['implied_prob']*100:.1f}%)")

## Optional API cross-check

This section is off by default. Choose a live mode and enable `RUN_EXTRA_API_CHECKS`
to send the selected leg prices to the API for comparison with the local calculation.
Keep account observations private. The local math above works without this request.


In [ ]:
check_legs = [leg["price"] for leg in legs] if len(legs) >= 2 else [-110, -110, -110]
if MODE == "offline" or not RUN_EXTRA_API_CHECKS:
    print("Optional API cross-check is off. The local parlay math above remains runnable.")
else:
    try:
        api = request_json("/v1/try/sgp/price", method="POST",
                           body={"legs": [{"price": value} for value in check_legs]})["independent_baseline"]
        ours = parlay(check_legs)
        assert abs(ours["combined_decimal"] - api["decimal_price"]) < 0.01
        print("The API cross-check agrees within rounding.")
    except (RuntimeError, KeyError, TypeError):
        print("API cross-check failed. No automatic retry.")


## The correlation trap

Everything above multiplies probabilities, which is only valid when the legs are
**independent**. Same-game legs usually are not:

- A quarterback's passing yards over and his team's win are positively
  correlated: parlaying them at the independent price underpays you if the book
  priced them independently, so books do not; they price the correlation in.
- A favorite's moneyline and the game total can be correlated through game
  script (a blowout tends to mean more possessions for one side).
- Even cross-game legs can correlate (weather systems, shared referees are weak
  examples; same-division doubleheaders less so).

Practical rules:

1. For cross-game parlays of unrelated events, independent multiplication is the
   right baseline, and everything in this notebook applies as-is.
2. For same-game parlays, the independent product is only a **baseline**. Real
   SGP prices move away from it, in either direction, depending on the
   correlation between the legs. That is why the keyless endpoint labels itself
   an independent baseline and why the keyed `/v1/sgp/price` endpoint exists: it
   applies literature-based correlation adjustments per leg pair. See
   [the docs](https://parlay-api.com/docs).
3. The vig compounding from earlier applies either way: every leg you add
   multiplies the book's margin into your price. Devig each leg (notebook 02)
   before deciding a parlay is worth it.

---

**More ParlayAPI resources**

- Docs: [parlay-api.com/docs](https://parlay-api.com/docs)
- Free API key (no card): [parlay-api.com/signup](https://parlay-api.com/signup)
- Browser calculators the math here matches: [no-vig](https://parlay-api.com/tools/no-vig-calculator), [parlay](https://parlay-api.com/tools/parlay-calculator), [EV](https://parlay-api.com/tools/ev-calculator)
- The rest of this series: [github.com/JacobiusMakes/parlayapi-notebooks](https://github.com/JacobiusMakes/parlayapi-notebooks)

These notebooks are for personal and internal research and education. Nothing here is betting advice.

## Private runtime data

Keep this notebook's code shareable and your account work private. Do not paste keys into
code cells, save them in notebook text, or commit downloaded observations. The hidden prompt
keeps the key in this runtime only. Clear all outputs before sharing or saving to GitHub;
Colab's output-omission setting is an additional safeguard, not a guarantee on other hosts.
The original published notebook contains no saved API results. Do not share an executed
account notebook or its exports. Each person uses their own account and key.

The MIT license covers code. API access does not grant public redisplay or redistribution
rights. Your applicable [terms](https://parlay-api.com/terms) and written agreement govern data.
Current coverage and plans: [docs](https://parlay-api.com/docs), [pricing](https://parlay-api.com/pricing).


In [ ]:
# Drop the runtime reference when finished. Restart the runtime to release other state.
_runtime_key = None
